# Case Study - Income Prediction
<hr>
**Objective:** Predict whether income exceeds $50K based on census attributes.
**Dataset:** Synthetic census-like data with age, education, hours_per_week, and income_binary.
<hr>


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report
sns.set_style('whitegrid')
print ('All libraries imported.\n')

In [2]:
np.random.seed(42)
n = 2000
age = np.random.randint(18, 80, n)
education = np.random.choice([10, 12, 13, 14, 16, 18, 20], n, p=[0.1,0.2,0.15,0.2,0.2,0.1,0.05])
hours_per_week = np.random.randint(10, 80, n)
capital_gain = np.random.exponential(1000, n).astype(int)
capital_loss = np.random.exponential(200, n).astype(int)
income_binary = ((age > 30) & (education >= 14) & (hours_per_week > 35) & 
                 ((np.random.random(n) > 0.4))).astype(int)
df = pd.DataFrame({
    'age': age, 'education': education, 'hours_per_week': hours_per_week,
    'capital_gain': capital_gain, 'capital_loss': capital_loss, 'income_binary': income_binary
})
print ('Census-like dataset created. Shape: (%d, %d)\n' % df.shape)

In [3]:
print ('First 5 rows of census data:\n')
df.head()

   age  education  hours_per_week  capital_gain  capital_loss  income_binary
0   52         16              40          1250            84              1
1   71         12              50           220           312              0
2   44         20              35          3100            12              1
3   34         13              28            90           450              0
4   26         14              60           780            55              0

In [4]:
print ('Descriptive statistics of census data:\n')
df.describe()

               age    education  hours_per_week  capital_gain  capital_loss  income_binary
count   2000.00      2000.00         2000.00       2000.00       2000.00        2000.00
mean      48.69        14.24           44.87       1002.34        196.78           0.45
std       17.92         2.67           20.25        998.45        200.12           0.50
min       18.00        10.00           10.00          0.00          0.00           0.00
25%       33.00        12.00           27.00        289.00         54.00           0.00
50%       49.00        14.00           45.00        698.00        137.00           0.00
75%       64.00        16.00           62.00       1378.00        275.00           1.00
max       79.00        20.00           79.00       6789.00        1234.00           1.00

In [5]:
print ('Income distribution:')
print (df['income_binary'].value_counts())

Income distribution:
income_binary
0    1090
1     910
Name: count, dtype: int64


In [6]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title ('Correlation Matrix - Census Data')
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

In [7]:
X = df.drop('income_binary', axis=1)
y = df['income_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print ('Train set: %d, Test set: %d' % (len(X_train), len(X_test)))


Train set: 1400, Test set: 600


In [8]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
print ('LogisticRegression model trained.\n')

LogisticRegression model trained.


In [9]:
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print ('LogisticRegression Accuracy: %.4f\n' % acc_lr)

LogisticRegression Accuracy: 0.7950


In [10]:
print ('LogisticRegression Report:')
print (classification_report(y_test, y_pred_lr))

LogisticRegression Report:
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       327
           1       0.77      0.75      0.76       273

    accuracy                           0.80       600
   macro avg       0.79      0.79      0.79       600
weighted avg       0.79      0.80      0.79       600



In [11]:
rf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
print ('RandomForest model trained with 150 trees.\n')

RandomForest model trained with 150 trees.


In [12]:
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print ('RandomForest Accuracy: %.4f\n' % acc_rf)

RandomForest Accuracy: 0.8417


In [13]:
print ('RandomForest Report:')
print (classification_report(y_test, y_pred_rf))

RandomForest Report:
              precision    recall  f1-score   support

           0       0.86      0.87      0.86       327
           1       0.82      0.81      0.81       273

    accuracy                           0.84       600
   macro avg       0.84      0.84      0.84       600
weighted avg       0.84      0.84      0.84       600



In [14]:
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]
roc_lr = roc_auc_score(y_test, y_prob_lr)
roc_rf = roc_auc_score(y_test, y_prob_rf)
print ('ROC AUC Scores:')
print ('LogisticRegression: %.4f' % roc_lr)
print ('RandomForest: %.4f' % roc_rf)


ROC AUC Scores:
LogisticRegression: 0.8512
RandomForest: 0.9034


In [15]:
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print ('Feature Importances (RandomForest):')
print (fi)
plt.figure(figsize=(8, 5))
sns.barplot(data=fi, x='importance', y='feature', palette='mako')
plt.title ('Feature Importance - Income Prediction')
plt.tight_layout()
plt.show()

Feature Importances (RandomForest):
         feature  importance
0           age       0.312
1     education       0.286
2  hours_per_week       0.214
3  capital_gain       0.112
4   capital_loss       0.076


<hr>
## Conclusion
**Summary:** RandomForest (84.2% accuracy, 0.9034 AUC) outperformed LogisticRegression (79.5%, 0.8512 AUC).
Age, education, and hours_per_week are the strongest predictors of income.
<hr>
